# Librerías

In [1]:
from dotenv import load_dotenv
import os
from google.genai import types
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from langchain_postgres import PGVector

# RAG

## Chunking

### Leemos los Documentos

In [ ]:
folder_root = "documentos"
documents = []
for folder in os.listdir(folder_root):
    if folder.endswith(".md"):
        path = os.path.join(folder_root, folder)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
            # Convertimos a clase Document para adjuntar fácilmente metadata
            doc = Document(
                page_content=text,
                metadata={"file_name": folder}
            )
            documents.append(doc)
    else:
        files_path = os.path.join(folder_root, folder)
        files = os.listdir(files_path)
        for file in files:
            path = os.path.join(files_path, file)
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
                # Convertimos a clase Document para adjuntar fácilmente metadata
                doc = Document(
                    page_content=text,
                    metadata={"file_name": file}
                )
                documents.append(doc)

len(documents)

### MarkdownHeaderTextSplitter

Este Splitter es muy útil para documentos Markdown, pues hace los splits según los headers que se encuentre

In [ ]:
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
    ("####", "h4"),
    ("#####", "h5"),
    ("######", "h6")
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on,
                                         strip_headers=False, # Si no queremos que el título de los headers aparezca en el chunk. Si False, la LLM no lee el título del chunk que está analizando
                                         return_each_line=False) # Si queremos que los \n\n se mantengan. Puede ser útil si los documentos siguen cierta jerarquía y rigurosidad (lo cual es difícil de mantener) para hacer luego otro splitting si el contenido es demasiado grande

chunked_documents = []
for doc in documents:
    chunk_id = 1
    # Hacemos el chunking
    chunks = md_splitter.split_text(doc.page_content) # Esto me da objetos de clase Document, pero el metadata del documento original se pierde
    for chunk in chunks:
        chunk.metadata = {**doc.metadata, "headers": list(chunk.metadata.values()), "chunk_id": chunk_id} # Ahora chunk contiene la metadata del documento original
        chunked_documents.append(chunk)
        chunk_id += 1
len(chunked_documents)

## Definimos el Embedding

In [16]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY2")

doc_embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001", 
    google_api_key = api_key,
    task_type="RETRIEVAL_DOCUMENT", # Gemini nos ofrece distintas formas de hacer el embedding para cada modelo, optimizado para la tarea que se requiera
    output_dimensionality=1024 # Dimensión del espacio vectorial sobre el que se definen los embedding
) 

## Añadimos los Chunks a la Base de Datos

https://docs.langchain.com/oss/python/integrations/embeddings

https://docs.langchain.com/oss/python/integrations/stores 

In [6]:
import os 
from langchain_classic.embeddings import CacheBackedEmbeddings  
from langchain_classic.storage import LocalFileStore

In [ ]:
ids = [i for i in range(1, len(chunked_documents) + 1)]
ids[-1]

In [ ]:
# Clase Store que sirve para guardar EN LOCAL los embeddings
store = LocalFileStore("./embeddings_guardados_todos")

# Creamos el cache
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings = doc_embedding,
    document_embedding_cache = store,
    namespace = "todos"
)

# Configuramos la base de datos con el cache
CONNECTION_STRING = os.getenv("CONNECTION_STRING")
# Nombre de la tabla que LangChain creará/usará
COLLECTION_NAME = "urbo_doc"

vector_store = PGVector(
    embeddings = cached_embedder,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION_STRING,
    use_jsonb=True, # Permite guardar metadatos extra muy fácilmente
)

# Leemos el punto de guardado (si existe)
archivo_guardado = "punto_de_guardado.txt"
if os.path.exists(archivo_guardado):
    with open(archivo_guardado, "r") as f:
        inicio = int(f.read().strip())
    print(f"Reanudamos desde el documento: {inicio}")
else:
    inicio = 0

# Preparamos los datos
ids = [str(i) for i in range(1, len(chunked_documents) + 1)]
#ids = [str(i) for i in range(815, 815 + len(chunked_documents))]
batch_size = 10

# Bucle de subida (empezando desde inicio)
for i in range(inicio, len(chunked_documents), batch_size):
    batch_docs = chunked_documents[i: i + batch_size]
    batch_ids = ids[i: i + batch_size]
    print("batch_docs: ", batch_docs)
    print("batch_ids: ", batch_ids)

    try:
        # Intentamos subir el batch a PostgreSQL
        vector_store.add_documents(batch_docs, ids = batch_ids)
        print(f"Lote insertado con éxito. Progreso: {i + len(batch_docs)} / {len(chunked_documents)}")
    except Exception as e:
        print(f"\n[ERROR o LÍMITE DE CUOTA ALCANZADO]")
        print(f"Falló al intentar procesar el lote que empieza en el documento {i}")
        print(f"Detalle: {e}")

        # Guardamos EXACTAMENTE por dónde íbamos por si salta error
        with open(archivo_guardado, "w") as f:
            f.write(str(i))
        break

else:
    print("\n¡Todos los documentos han sido subidos a PGVector con éxito!")
    if os.path.exists(archivo_guardado):
        os.remove(archivo_guardado) # Borramos el archivo de guardado porque ya terminamos